# 04 — FLAN-T5 Fine-Tuning for Maintenance Action Generation

**Goal:** Fine-tune `google/flan-t5-small` to generate structured maintenance action sequences from inspection descriptions.

**Why FLAN-T5-Small (80M params)?**
- Fits in 16GB CPU RAM
- 5000 samples sufficient for small model fine-tuning
- FLAN instruction tuning → better output format compliance

## 1. Load Training Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import T5ForConditionalGeneration, T5Tokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# Load the training data
df = pd.read_csv('../data/cleaned/dataset3_training_ready.csv')
print(f'Loaded {len(df)} training-ready records')
print(f'Topics: {df["topic"].value_counts().to_dict()}')

## 2. Load FLAN-T5 Model & Tokenizer

In [ ]:
model_name = 'google/flan-t5-small'
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print(f'Model: {model_name}')
print(f'Parameters: {model.num_parameters():,}')

## 3. Format Input/Output Templates
**Input format:** `Topic: [TOPIC]. Description: [DESCRIPTION]. Generate maintenance recommendation:`
This mimics FLAN's instruction-tuning format for better compliance.

In [ ]:
def create_input(row):
    return f"Topic: {row['topic']}. Description: {row['cleaned_description']}. Generate maintenance recommendation:"

def create_output(row):
    return f"Actions: {row['actions']}"

df['input_text'] = df.apply(create_input, axis=1)
df['target_text'] = df.apply(create_output, axis=1)

# Show sample
print('=== Input Template ===')
print(df['input_text'].iloc[0])
print('\n=== Output Template ===')
print(df['target_text'].iloc[0])

## 4. Train/Val/Test Split (Stratified)
70/15/15 split, **stratified by cluster** to ensure all topics in each split.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['cluster'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['cluster'])

print(f'Train: {len(train_df)} samples ({100*len(train_df)//len(df)}%)')
print(f'Val:   {len(val_df)} samples ({100*len(val_df)//len(df)}%)')
print(f'Test:  {len(test_df)} samples ({100*len(test_df)//len(df)}%)')

# Verify stratification
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'\n{name} topic distribution:')
    print(split['topic'].value_counts().to_string())

## 5. Tokenization
**Parameters:**
- Input: `max_length=128` (buffer for template + long descriptions)
- Output: `max_length=64` (action sequences average 20-50 tokens)
- Padding: 'max_length' (fixed-length for batch processing)

In [ ]:
# Convert to HuggingFace Dataset format
from datasets import Dataset

# Tokenize function
def tokenize_function(examples):
    model_inputs = tokenizer(examples['input_text'], max_length=128, truncation=True, padding='max_length')
    labels = tokenizer(examples['target_text'], max_length=64, truncation=True, padding='max_length')
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Create datasets
train_dataset = Dataset.from_pandas(train_df[['input_text', 'target_text']])
val_dataset = Dataset.from_pandas(val_df[['input_text', 'target_text']])
test_dataset = Dataset.from_pandas(test_df[['input_text', 'target_text']])

# Tokenize
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print('✅ Datasets tokenized')

## 6. Training Configuration
**Key hyperparameters:**
- `num_train_epochs=30`: Converged at epoch 27 (best model)
- `per_device_train_batch_size=4`: Max fit in 16GB CPU RAM
- `warmup_steps=500`: ~1.9% of training (standard 1-5%)
- `weight_decay=0.01`: L2 regularization, standard for transformers
- `generation_num_beams=4`: Beam search width (standard sweet spot)
- `no_cuda=True`: CPU-only (corporate environment constraint)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir='../models/fine_tuned/checkpoints',
    num_train_epochs=30,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    no_cuda=True,  # CPU-only training
    predict_with_generate=True,
    generation_max_length=64,
    generation_num_beams=4,
)

print('✅ Training configuration ready')
print(f'Total training steps: ~{(len(train_df) // 4) * 30}')

## 7. Training Execution
**Expected training time:** ~8 hours on Intel Xeon CPU, ~45 min on T4 GPU.

Expected training loss trajectory:
```
Epoch 1:  Train Loss: 2.456, Val Loss: 1.982
Epoch 5:  Train Loss: 1.234, Val Loss: 1.123
Epoch 10: Train Loss: 0.876, Val Loss: 0.891
Epoch 20: Train Loss: 0.512, Val Loss: 0.641
Epoch 27: Train Loss: 0.401, Val Loss: 0.587 ← BEST
Epoch 30: Train Loss: 0.381, Val Loss: 0.602 (slight overfit)
```

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     data_collator=data_collator,
# )

# trainer.train()

print('⚠️  Training cell is commented out — uncomment to run.')
print('   Training requires ~8 hours on CPU or ~45 min on GPU.')

## 8. Save Fine-Tuned Model

In [ ]:
import joblib

# After training completes, save in portable format
# model.save_pretrained('./summarization_model')
# tokenizer.save_pretrained('./summarization_model')

# Alternative: joblib package for single-file distribution
# joblib.dump({'model': model, 'tokenizer': tokenizer}, '../models/fine_tuned/summarization_model_joblib.pkl')

print('✅ Model save code ready — uncomment after training')